# Batch 3 — Reproduction of exp_009 (inference only, no training)

**Base paper:** H. Rahman, *Fine-tuning Whisper for Pashto ASR: strategies and scale*, arXiv:2604.06507 (7 Apr 2026), DOI 10.48550/arXiv.2604.06507.

This notebook runs the three cheapest and most important experiments of the assignment. **Nothing here trains a model**, so it cannot fail the way a 9-hour training run can.

| Part | What it does | Paper's reported value |
|---|---|---|
| **B0** | Zero-shot `openai/whisper-small` on the Common Voice v24 Pashto test split | 120.4 % WER |
| **B** | The author's own released checkpoint `ihanif/exp_009_small_cv24_vanilla`, scored by **our** harness | **24.89 % WER, 8.075 % CER** |
| **D** | Our two original analyses: script-failure rate, and a Pashto orthographic-normalisation ablation | not in the paper |

Part B is the row that answers *"did the paper reproduce?"*. Part B0 gives it something to be measured against. Part D is the contribution that is ours alone.

### Runs on both Colab and Kaggle
The notebook detects which one it is on. **Kaggle is the easier platform** because you can attach the Common Voice mirror as a dataset and skip the download entirely; on Colab you have to get the archive into Google Drive first. Cell 4 handles all three data routes.

### Before you start
Set `HF_DATASET_TOKEN` in cell 4 if you are using Route A, or attach the Kaggle dataset for Route B. Everything else runs as-is.

## 1. Environment record

Writes `results/environment.txt` and `results/gpu.txt`. These two files are required execution evidence for §5.1 of the report — do not transcribe the versions by hand, let this cell capture them.

In [ ]:
import os, sys, platform, subprocess, json, pathlib

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

ON_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")
ON_KAGGLE = os.path.exists("/kaggle/input") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
PLATFORM  = "Google Colab" if ON_COLAB else ("Kaggle" if ON_KAGGLE else "local")
print("platform:", PLATFORM)
print("python  :", sys.version.split()[0], "|", platform.platform())

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout or "NO GPU DETECTED -- enable one before continuing")
(RESULTS / "gpu.txt").write_text(gpu.stdout or "no gpu", encoding="utf-8")

if not gpu.stdout:
    print("\n>>> Colab:  Runtime -> Change runtime type -> T4 GPU")
    print(">>> Kaggle: Settings -> Accelerator -> GPU T4 x2 (or P100)")

## 2. Install pinned dependencies

Pins sit inside the version floors the paper states (Python 3.10, PyTorch 2.x/CUDA 12, transformers ≥ 4.40, datasets ≥ 2.18, jiwer v3).

One pin is load-bearing: **`datasets` must stay on the 2.x line.** Version 3.x removed script-based loading and changed the `Audio` feature API, which breaks this pipeline.

On Colab this takes 2–3 minutes and you will be asked to restart the runtime. **Restart, then skip straight back to cell 3** — do not re-run cell 2.

In [ ]:
PINS = ["transformers==4.44.2", "datasets==2.21.0", "accelerate==0.33.0",
        "evaluate==0.4.3", "jiwer==3.0.4", "librosa==0.10.2", "soundfile==0.12.1"]

import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS],
                   capture_output=True, text=True)
print("pip exit code:", r.returncode)
print(r.stdout[-2500:] or "(quiet)")
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-3000:])
print("\nIf Colab asks you to restart the runtime: restart, then re-run from cell 3.")
print("Do NOT re-run this cell after restarting.")

In [ ]:
import sys, pathlib, subprocess
RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

import torch, transformers, datasets, evaluate, jiwer
env = "\n".join([
    f"platform              : {sys.platform}",
    f"python                : {sys.version.split()[0]}",
    f"torch                 : {torch.__version__}",
    f"cuda (torch)          : {torch.version.cuda}",
    f"gpu                   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}",
    f"gpu_total_memory_gb   : {round(torch.cuda.get_device_properties(0).total_memory/1e9,2) if torch.cuda.is_available() else 0}",
    f"transformers          : {transformers.__version__}",
    f"datasets              : {datasets.__version__}",
    f"evaluate              : {evaluate.__version__}",
    f"jiwer                 : {jiwer.__version__}",
])
print(env)
(RESULTS / "environment.txt").write_text(env, encoding="utf-8")
with open(RESULTS / "pip_freeze.txt", "w", encoding="utf-8") as fh:
    fh.write(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                            capture_output=True, text=True).stdout)
print("\nwrote results/environment.txt, results/gpu.txt, results/pip_freeze.txt")

## 3. Recover the author's actual configuration from his published log

**Read this section — it changes what you write in the report.**

The paper never states its optimizer, and leaves warmup, batch composition and several other settings unstated for exp_009. But the author's Hugging Face checkpoint repository ships two files the paper does not discuss:

- `experiment_log.txt` (3.9 MB) — the **complete training log** of exp_009, beginning with a full `ARGUMENTS SUMMARY`
- `training_args.bin` — the serialised HuggingFace `TrainingArguments` object

Both are public and ungated. This cell downloads them, prints the recovered configuration, and extracts the paper's own evaluation curve into a CSV you can plot against yours.

Three things this recovers that the paper does not publish:

1. **`eval_dataset_splits: ['test']`** — the evaluation loop that selected the best checkpoint ran on the **test split**. The paper says the best checkpoint was chosen by "normalised validation WER"; the log shows that "validation" *was* the test set. The reported 24.89 % is therefore selected on the same data it is reported on. Say this in §9 of the report — it is a genuine methodological finding, and it means our held-out-dev setup is cleaner than the paper's.
2. **`fp16: False`, `bf16: False`, `train_batchsize: 64`, `gradient_accumulation_steps: 1`** — FP32 at a true batch of 64, confirming why free-tier hardware cannot match it.
3. **`warmup_steps: 500`, `weight_decay: 0.0`, `early_stopping_patience: 5`, `min_duration_in_seconds: 0.5`** — values the paper omits. Note that the duration floor is 0.5 s, not the 1.0 s used in our own `prepare_data.py`.

In [ ]:
import re, csv, urllib.request, pathlib, json

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)
PAPER_DIR = RESULTS / "paper_artifacts"; PAPER_DIR.mkdir(exist_ok=True)
CKPT = "ihanif/exp_009_small_cv24_vanilla"
BASE = f"https://huggingface.co/{CKPT}/resolve/main"

for fname in ("experiment_log.txt", "training_args.bin", "config.json"):
    dest = PAPER_DIR / fname
    if not dest.exists():
        urllib.request.urlretrieve(f"{BASE}/{fname}", dest)
    print(f"{fname:24s} {dest.stat().st_size:>10,} bytes")

log = (PAPER_DIR / "experiment_log.txt").read_text(encoding="utf-8", errors="replace")

# --- the ARGUMENTS SUMMARY block ---------------------------------------
block = log.split("=" * 70)
args_text = block[2] if len(block) > 2 else log[:6000]
print("\n" + "=" * 72)
print("AUTHOR'S PUBLISHED CONFIGURATION FOR exp_009")
print("=" * 72)
print(args_text.strip()[:4000])

author_args = dict(re.findall(r"^([a-z_]+)\s+:\s(.*)$", args_text, flags=re.M))
(PAPER_DIR / "author_exp009_args.json").write_text(
    json.dumps(author_args, indent=2), encoding="utf-8")

print("\n" + "=" * 72)
print("SETTINGS THE PAPER DOES NOT STATE, RECOVERED HERE")
print("=" * 72)
for k in ["eval_dataset_splits", "train_dataset_splits", "learning_rate", "train_batchsize",
          "gradient_accumulation_steps", "warmup_steps", "warmup_ratio", "weight_decay",
          "fp16", "bf16", "num_epochs", "eval_steps", "save_steps", "save_total_limit",
          "early_stopping_patience", "metric_for_best_model", "gradient_checkpointing",
          "min_duration_in_seconds", "max_duration_in_seconds", "use_lora",
          "freeze_encoder_layers", "augment_speed", "augment_noise"]:
    if k in author_args:
        print(f"  {k:30s} = {author_args[k]}")
print("\n  optimizer                      = not passed anywhere, so the HuggingFace")
print("                                   Seq2SeqTrainer default (adamw_torch) was used")

In [ ]:
# --- the paper's own evaluation curve, recovered from the same log -----
rows = []
for m in re.finditer(r"\{'eval_loss':.*?\}", log):
    rows.append(dict(re.findall(r"'([a-z_]+)':\s*'([^']*)'", m.group(0))))

keys = ["epoch", "eval_loss", "eval_wer", "eval_wer_ortho", "eval_cer"]
with open(RESULTS / "paper_exp009_eval_curve.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=keys); w.writeheader()
    for r in rows:
        w.writerow({k: r.get(k, "") for k in keys})

print(f"{len(rows)} evaluation points recovered -> results/paper_exp009_eval_curve.csv\n")
print(f"{'epoch':>7} {'loss':>8} {'WER':>7} {'WER_orth':>9} {'CER':>7}")
for r in rows:
    print(f"{float(r['epoch']):7.3f} {float(r['eval_loss']):8.4f} "
          f"{float(r['eval_wer']):7.2f} {float(r['eval_wer_ortho']):9.2f} "
          f"{float(r['eval_cer']):7.3f}")

best = min(rows, key=lambda r: float(r["eval_wer"]))
lo_loss = min(rows, key=lambda r: float(r["eval_loss"]))
print(f"\nbest WER  : {best['eval_wer']}% at epoch {best['epoch']} "
      f"(eval_loss {best['eval_loss']})  <- the paper's headline 24.89%")
print(f"lowest loss: {lo_loss['eval_loss']} at epoch {lo_loss['epoch']} "
      f"(WER {lo_loss['eval_wer']}%)")
print("\nNote for section 9 of the report: the lowest validation loss and the best WER")
print("occur at DIFFERENT epochs, and the loss RISES while the WER keeps improving.")
print("That is why the paper selects on WER rather than on loss, and our own curves")
print("should be checked for the same decoupling.")

## 4. Get the Common Voice v24 Pashto **test** split

Three routes. Try them in this order; set `DATA_ROUTE` below.

### Route A — the author's Hugging Face dataset (best if it works)
`ihanif/common-voice-pashto-24` is the **exact** dataset object the paper trained and evaluated on, so it gives perfect comparability. Anonymous access returns HTTP 401, and Hugging Face reports it as *"private or gated"* — which of the two it is cannot be determined without logging in.

**Spend two minutes checking:** open <https://huggingface.co/datasets/ihanif/common-voice-pashto-24> while signed in to a free HF account.
- If you see an **"Agree and access repository"** button → it is *gated*. Click it, then create a read token at <https://huggingface.co/settings/tokens> and paste it below. Route A works.
- If you see **404 / "Repository not found"** → it is *private*. Route A is dead; use Route B.

### Route B — Kaggle mirror (the reliable fallback)
A CC0-1.0 mirror of the official release: `ataullahaali/common-voice-scripted-speech-24-0-pashto`. On Kaggle, *Add Input → Datasets → search that slug*; it mounts read-only under `/kaggle/input/` with **no download at all**. This is why Kaggle is the easier platform for this notebook.

Cite it in the report **as an unofficial mirror**, with the official Mozilla Data Collective page as the primary source.

### Route C — official Mozilla archive via Google Drive
Download Common Voice Scripted Speech 24.0, locale `ps`, from <https://datacollective.mozillafoundation.org/datasets/cmj8u3pnb00llnxxbfvxo3b14> (free account, CC0-1.0), put the archive in your Drive, and point `DRIVE_ARCHIVE` at it. Most authentic, slowest.

In [ ]:
# ======================= CONFIGURE THIS CELL =========================
DATA_ROUTE = "B"          # "A" = HF gated dataset | "B" = Kaggle mirror | "C" = Drive archive

HF_DATASET_TOKEN = ""     # Route A only: paste a read token from huggingface.co/settings/tokens
KAGGLE_INPUT     = "/kaggle/input/common-voice-scripted-speech-24-0-pashto"   # Route B
DRIVE_ARCHIVE    = "/content/drive/MyDrive/cv-corpus-24.0-ps.tar.gz"          # Route C

# Start with a seeded subsample so you get numbers in ~15 min and can confirm the
# pipeline is sound. Set to None for the FINAL numbers that go in the report --
# the paper used all 13,791 utterances, and the report must state which you used.
MAX_EVAL_SAMPLES = 2000   # None = the full official test split

SEED = 42
# =====================================================================
PAPER = {"wer": 24.89, "wer_ortho": 27.73, "cer": 8.075,
         "zeroshot_wer": 120.4, "test_rows": 13791, "test_hours": 20.72}
print(f"route={DATA_ROUTE}  max_eval_samples={MAX_EVAL_SAMPLES}  seed={SEED}")

In [ ]:
import os, glob, pathlib, random, json, subprocess
import pandas as pd

random.seed(SEED)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
test_rows = None          # will become a DataFrame with columns: audio, sentence, client_id
hf_test  = None           # or a HuggingFace Dataset, for route A

# ---------------------------------------------------------------- ROUTE A
if DATA_ROUTE == "A":
    from datasets import load_dataset, Audio
    assert HF_DATASET_TOKEN, "paste a read token into HF_DATASET_TOKEN first"
    hf_test = load_dataset("ihanif/common-voice-pashto-24", split="test",
                           token=HF_DATASET_TOKEN)
    hf_test = hf_test.cast_column("audio", Audio(sampling_rate=16000))
    print("loaded the author's own test split:", len(hf_test), "rows")
    print("columns:", hf_test.column_names)

# ---------------------------------------------------------------- ROUTE B
elif DATA_ROUTE == "B":
    hits = glob.glob(os.path.join(KAGGLE_INPUT, "**", "test.tsv"), recursive=True)
    assert hits, (f"no test.tsv under {KAGGLE_INPUT}\n"
                  "Attach the Kaggle dataset 'ataullahaali/common-voice-scripted-speech-24-0-pashto' "
                  "via Add Input -> Datasets, then re-run this cell.")
    CV_ROOT = pathlib.Path(hits[0]).parent
    print("Common Voice root:", CV_ROOT)
    print("files:", sorted(p.name for p in CV_ROOT.iterdir() if p.is_file())[:12])

# ---------------------------------------------------------------- ROUTE C
elif DATA_ROUTE == "C":
    from google.colab import drive
    drive.mount("/content/drive")
    assert os.path.exists(DRIVE_ARCHIVE), f"not found: {DRIVE_ARCHIVE}"
    os.makedirs("/content/cv", exist_ok=True)
    print("extracting -- this takes a while for a multi-gigabyte archive")
    subprocess.run(["tar", "-xf", DRIVE_ARCHIVE, "-C", "/content/cv"], check=True)
    hits = glob.glob("/content/cv/**/test.tsv", recursive=True)
    assert hits, "test.tsv not found inside the archive"
    CV_ROOT = pathlib.Path(hits[0]).parent
    print("Common Voice root:", CV_ROOT)

else:
    raise ValueError("DATA_ROUTE must be A, B or C")

## 5. Build the test manifest and apply the paper's filters

Filters are taken from the author's recovered configuration, not guessed: `min_duration_in_seconds: 0.5`, `max_duration_in_seconds: 30.0`. The split itself is the **official shipped `test.tsv`** — no re-splitting, because Common Voice builds its splits to be speaker-disjoint and a custom re-split would leak speakers and inflate the score.

Compare the printed row count against the paper's 13,791. A mismatch is a finding to report in §4.2, not a bug to hide.

In [ ]:
import csv as _csv, pathlib, json
import pandas as pd

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)
stats = {"route": DATA_ROUTE, "seed": SEED, "paper_reference": PAPER}

if DATA_ROUTE == "A":
    df = pd.DataFrame({
        "sentence":  hf_test["sentence"],
        "client_id": hf_test["client_id"] if "client_id" in hf_test.column_names
                     else ["unknown"] * len(hf_test),
    })
    df["audio"] = list(range(len(df)))          # index into hf_test
    n_raw = len(df)
    print(f"rows in the author's test split: {n_raw}  (paper: {PAPER['test_rows']})")
    durations = None

else:
    durs = {}
    dur_tsv = CV_ROOT / "clip_durations.tsv"
    if dur_tsv.exists():
        d = pd.read_csv(dur_tsv, sep="\t")
        durs = dict(zip(d[d.columns[0]], d[d.columns[1]] / 1000.0))
        print(f"clip_durations.tsv: {len(durs)} entries")
    else:
        print("clip_durations.tsv absent -- durations will be probed with soundfile")

    df = pd.read_csv(CV_ROOT / "test.tsv", sep="\t", quoting=_csv.QUOTE_NONE,
                     dtype=str, keep_default_na=False)
    n_raw = len(df)
    print(f"rows in official test.tsv: {n_raw}  (paper: {PAPER['test_rows']})")

    clips = CV_ROOT / "clips"
    df["audio"] = [str(clips / p) for p in df["path"]]
    if durs:
        df["duration_s"] = [durs.get(p, float("nan")) for p in df["path"]]
    else:
        import soundfile as sf
        def _probe(p):
            try:
                i = sf.info(p); return i.frames / i.samplerate
            except Exception:
                return float("nan")
        df["duration_s"] = [_probe(a) for a in df["audio"]]

# ---- filters -----------------------------------------------------------
steps = []
def drop(frame, rule, mask, source):
    before = len(frame)
    frame = frame[mask].copy()
    steps.append({"rule": rule, "rows_in": before, "rows_out": len(frame),
                  "removed": before - len(frame), "source": source})
    print(f"  {rule:48s} {before:>7} -> {len(frame):>7}  (-{before-len(frame)})")
    return frame

print("\nfilters:")
df = drop(df, "drop empty transcripts", df["sentence"].str.strip() != "", "ours")
if "duration_s" in df:
    df = drop(df, "duration in [0.5, 30.0] s",
              df["duration_s"].between(0.5, 30.0, inclusive="both"),
              "author's recovered config")

total_h = df["duration_s"].sum() / 3600.0 if "duration_s" in df else float("nan")
print(f"\nkept {len(df)} utterances"
      + (f" / {total_h:.2f} h  (paper: {PAPER['test_rows']} / {PAPER['test_hours']} h)"
         if "duration_s" in df else ""))

n_full = len(df)
if MAX_EVAL_SAMPLES and MAX_EVAL_SAMPLES < len(df):
    df = df.sample(n=MAX_EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)
    print(f"\n*** SEEDED SUBSAMPLE: {len(df)} of {n_full} utterances ***")
    print("*** For the numbers that go in the report, set MAX_EVAL_SAMPLES = None ***")
    print("*** and state in section 4.2 which set each figure was measured on.    ***")
else:
    df = df.reset_index(drop=True)
    print("\nusing the FULL official test split -- directly comparable to the paper")

if "client_id" in df:
    print(f"unique speakers in the evaluated set: {df['client_id'].nunique()}")

stats.update({"rows_raw": int(n_raw), "rows_after_filters": int(n_full),
              "rows_evaluated": int(len(df)),
              "hours_after_filters": None if "duration_s" not in df else round(float(total_h), 3),
              "speakers_evaluated": int(df["client_id"].nunique()) if "client_id" in df else None,
              "filter_steps": steps})
(RESULTS / "dataset_stats_batch3.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
pd.DataFrame(steps).to_csv(RESULTS / "preprocessing_log_batch3.csv", index=False)
print("\nwrote results/dataset_stats_batch3.json and results/preprocessing_log_batch3.csv")
TEST = df

## 6. Metrics

Three WER variants, because quoting one without labelling it is ambiguous:

- **orthographic WER** — raw strings
- **normalised WER** — both sides through HuggingFace `BasicTextNormalizer`. **This is the paper's headline metric**, so it is the one the difference calculation must use on both sides.
- **CER** — character level

Plus: the substitution/deletion/insertion split (to check the paper's claim that substitutions are ~70–78 % of word errors), a bootstrap 95 % confidence interval, the **Pashto-script output rate**, and the Pashto canonicalisation map used for the normalisation ablation.

Why the ablation matters: `BasicTextNormalizer` lowercases and strips punctuation but performs **no Arabic-script canonicalisation**. It will not unify Arabic yeh `ي` (U+064A) with Pashto yeh `ی` (U+06CC), nor Arabic kaf `ك` with Pashto kaf `ک`, and it leaves the zero-width non-joiner, tatweel and harakat in place. All are frequent, legitimate Pashto spelling variants, so part of the paper's 24.89 % is spelling disagreement rather than recognition error. Nobody has measured how much. We do, in cell 9.

In [ ]:
import unicodedata, numpy as np, evaluate as hf_evaluate, jiwer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

_norm = BasicTextNormalizer()
_wer  = hf_evaluate.load("wer")
_cer  = hf_evaluate.load("cer")

# Pashto-distinctive letters. Their presence means genuine Pashto orthography;
# their absence usually means the model fell back to Arabic / Dari / Urdu script.
PASHTO_DISTINCTIVE = set("ټډړڼږښځڅګۍ")

_CANON = {"ي": "ی",  # Arabic yeh    -> Pashto yeh
          "ى": "ی",  # alef maksura  -> Pashto yeh
          "ك": "ک",  # Arabic kaf    -> Pashto kaf
          "ۀ": "ه",  # heh+hamza     -> heh
          "ة": "ه"}  # teh marbuta   -> heh
_STRIP = ("‌‍​"                                     # ZWNJ, ZWJ, ZWSP
          "ـ"                                                  # tatweel
          "ًٌٍَُِّْٰ"  # harakat
          "،؛؟۔")                               # Arabic punctuation
_DIGITS = {**{chr(0x0660 + i): str(i) for i in range(10)},
           **{chr(0x06F0 + i): str(i) for i in range(10)}}

def pashto_canonical(text):
    """Used ONLY for the ablation. Never touches the distinctive letters."""
    for a, b in _CANON.items():
        text = text.replace(a, b)
    text = "".join(_DIGITS.get(c, c) for c in text if c not in _STRIP)
    return " ".join(unicodedata.normalize("NFC", text).split())

def has_pashto_script(text):
    return any(c in PASHTO_DISTINCTIVE for c in text)

def score(references, hypotheses, extra_norm=None):
    """extra_norm=pashto_canonical runs the ablation on identical predictions."""
    ref_n, hyp_n, ref_o, hyp_o = [], [], [], []
    for r, h in zip(references, hypotheses):
        if extra_norm:
            r, h = extra_norm(r), extra_norm(h)
        rn, hn = _norm(r).strip(), _norm(h).strip()
        if not rn:
            continue                      # an empty reference makes WER undefined
        ref_n.append(rn); hyp_n.append(hn or " ")
        ref_o.append(r);  hyp_o.append(h or " ")

    out = jiwer.process_words(ref_n, hyp_n)
    err = out.substitutions + out.deletions + out.insertions
    return {
        "wer":       100 * _wer.compute(references=ref_n, predictions=hyp_n),
        "wer_ortho": 100 * _wer.compute(references=ref_o, predictions=hyp_o),
        "cer":       100 * _cer.compute(references=ref_n, predictions=hyp_n),
        "n":         len(ref_n),
        "sub": out.substitutions, "del": out.deletions, "ins": out.insertions,
        "sub_pct": round(100 * out.substitutions / err, 2) if err else 0,
        "del_pct": round(100 * out.deletions     / err, 2) if err else 0,
        "ins_pct": round(100 * out.insertions    / err, 2) if err else 0,
        "pashto_script_rate": 100 * float(np.mean([has_pashto_script(h) for h in hypotheses])),
    }

def bootstrap_wer_ci(references, hypotheses, n_boot=1000, seed=SEED):
    """Percentile CI by resampling utterances. WER is a ratio of summed counts,
    so it is not a proportion and Wilson intervals do not apply."""
    rng = np.random.default_rng(seed)
    pairs = [(r, h) for r, h in zip(references, hypotheses) if _norm(r).strip()]
    counts = []
    for r, h in pairs:
        o = jiwer.process_words(_norm(r).strip(), _norm(h).strip() or " ")
        counts.append((o.substitutions + o.deletions + o.insertions,
                       len(_norm(r).strip().split())))
    counts = np.array(counts, dtype=float)
    idx = rng.integers(0, len(counts), size=(n_boot, len(counts)))
    draws = 100 * counts[idx, 0].sum(axis=1) / counts[idx, 1].sum(axis=1)
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

def paired_bootstrap(references, hyp_a, hyp_b, n_boot=1000, seed=SEED):
    """One-sided p that system B is not better than system A, on the same utterances."""
    rng = np.random.default_rng(seed)
    rows = []
    for r, a, b in zip(references, hyp_a, hyp_b):
        rn = _norm(r).strip()
        if not rn:
            continue
        oa = jiwer.process_words(rn, _norm(a).strip() or " ")
        ob = jiwer.process_words(rn, _norm(b).strip() or " ")
        rows.append((oa.substitutions + oa.deletions + oa.insertions,
                     ob.substitutions + ob.deletions + ob.insertions,
                     len(rn.split())))
    arr = np.array(rows, dtype=float)
    idx = rng.integers(0, len(arr), size=(n_boot, len(arr)))
    wa = arr[idx, 0].sum(axis=1) / arr[idx, 2].sum(axis=1)
    wb = arr[idx, 1].sum(axis=1) / arr[idx, 2].sum(axis=1)
    return float(np.mean(wb >= wa))

print("metrics ready: score(), bootstrap_wer_ci(), paired_bootstrap()")
print("sanity check -- an empty hypothesis must give exactly 100% WER (all deletions):")
print("   ", {k: round(v, 2) if isinstance(v, float) else v
              for k, v in score(["سلام وروره"], [""]).items()})

## 7. Decoding

One function, used for every system, so the comparison is valid. The paper does not state its beam width or maximum generation length, so we fix **greedy decoding** and `max_new_tokens=225` (the HuggingFace Whisper default) and apply them identically to B0 and B. **Record this choice in §5.3 of the report** — beam search typically moves WER by 0.5–1.5 pp, so an unstated mismatch between two systems would invalidate the whole comparison.

`language="ps"` is forced on every call. This is the switch the paper shows is broken in zero-shot mode: the model accepts it and emits Arabic/Dari/Urdu script anyway.

In [ ]:
import torch, numpy as np, time, gc, librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
BATCH  = 16          # drop to 8 if you hit CUDA OOM on a T4

def load_audio(entry):
    if DATA_ROUTE == "A":
        return np.asarray(hf_test[int(entry)]["audio"]["array"], dtype=np.float32)
    a, _ = librosa.load(entry, sr=16000, mono=True)   # 16 kHz mono, Whisper's requirement
    return a.astype(np.float32)

def transcribe_all(model_id, frame, tag, num_beams=1, max_new_tokens=225):
    print(f"\n=== {tag}  ({model_id})")
    processor = WhisperProcessor.from_pretrained(model_id, language="ps", task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=DTYPE, low_cpu_mem_usage=True).to(DEVICE).eval()
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    print(f"    parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

    hyps, t0 = [], time.time()
    for start in range(0, len(frame), BATCH):
        chunk = frame.iloc[start:start + BATCH]
        feats = processor.feature_extractor(
            [load_audio(e) for e in chunk["audio"]],
            sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE, DTYPE)
        with torch.no_grad():
            ids = model.generate(feats, language="ps", task="transcribe",
                                 num_beams=num_beams, max_new_tokens=max_new_tokens)
        hyps.extend(processor.tokenizer.batch_decode(ids, skip_special_tokens=True))

        done = start + len(chunk)
        if start % (BATCH * 20) == 0 or done == len(frame):
            el = time.time() - t0
            print(f"    {done:>6}/{len(frame)}  {el:>6.0f}s elapsed  "
                  f"eta {el / max(done,1) * (len(frame) - done):>6.0f}s", flush=True)

    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"    done in {time.time()-t0:.0f}s")
    return [h.strip() for h in hyps]

print(f"device={DEVICE}  dtype={DTYPE}  batch={BATCH}  greedy decoding, max_new_tokens=225")

## 8. Part B0 — zero-shot baseline

Untouched `openai/whisper-small`. Paper's reported value: **120.4 % WER**.

Expect WER well above 100 %. That is not a bug — insertions are unbounded and are not normalised by reference length, so a model that fluently generates text in the wrong script accumulates insertions without limit. A system that output *nothing at all* would score exactly 100 %, so zero-shot Whisper is **worse than silence** on Pashto. That sentence belongs in your Discussion.

This is the slowest cell, because the zero-shot model produces long degenerate outputs. ~15 min for 2,000 utterances on a T4; ~1.5 h for the full split.

In [ ]:
import pandas as pd, pathlib
RESULTS = pathlib.Path("results")
REFS = list(TEST["sentence"])

hyp_zs = transcribe_all("openai/whisper-small", TEST, "B0 zero-shot whisper-small")
m_zs = score(REFS, hyp_zs)
ci_zs = bootstrap_wer_ci(REFS, hyp_zs)

print(f"\n{'':<22}{'ours':>10}{'paper':>10}{'diff (pp)':>12}")
print(f"{'WER (normalised)':<22}{m_zs['wer']:>10.2f}{PAPER['zeroshot_wer']:>10.2f}"
      f"{m_zs['wer']-PAPER['zeroshot_wer']:>+12.2f}")
print(f"{'WER (orthographic)':<22}{m_zs['wer_ortho']:>10.2f}{'n/a':>10}{'':>12}")
print(f"{'CER':<22}{m_zs['cer']:>10.2f}{'n/a':>10}{'':>12}")
print(f"\n95% bootstrap CI on WER: {ci_zs[0]:.2f} - {ci_zs[1]:.2f}")
print(f"errors: S {m_zs['sub']} / D {m_zs['del']} / I {m_zs['ins']}"
      f"   -> insertions are {m_zs['ins_pct']}% of all errors")
print(f"Pashto-script output rate: {m_zs['pashto_script_rate']:.2f}%"
      f"   (companion paper arXiv:2604.04598 reports < 0.8%)")

pd.DataFrame({"reference": REFS, "hypothesis": hyp_zs}).to_csv(
    RESULTS / "predictions_zeroshot_small.csv", index=False, encoding="utf-8")

print("\nfirst three outputs -- check which script they are in:")
for r, h in list(zip(REFS, hyp_zs))[:3]:
    print(f"  REF: {r}")
    print(f"  HYP: {h[:160]}")
    print(f"       contains Pashto-distinctive letters: {has_pashto_script(h)}\n")

## 9. Part B — the reproduction

The author's released checkpoint `ihanif/exp_009_small_cv24_vanilla`, scored by **our** harness on **our** reconstruction of the test split. Paper's reported values: **24.89 % WER, 8.075 % CER**.

This is the single most important cell in the project, because of what it isolates. If this agrees with 24.89 %, then our measurement pipeline is correct, and any gap in your own training run (Batch 4) is attributable to training compute rather than to a bug in how you compute WER. That lets the report make two separate, defensible claims instead of one vague one:

- **(a)** the paper's *reported result* is independently verifiable — settled here;
- **(b)** the paper's *training recipe* reproduces at our compute budget — settled in Batch 4.

In [ ]:
hyp_ck = transcribe_all(CKPT, TEST, "B author's exp_009 checkpoint")
m_ck  = score(REFS, hyp_ck)
ci_ck = bootstrap_wer_ci(REFS, hyp_ck)

print(f"\n{'':<22}{'ours':>10}{'paper':>10}{'diff (pp)':>12}{'rel %':>10}")
for label, key, ref in (("WER (normalised)", "wer", PAPER["wer"]),
                        ("WER (orthographic)", "wer_ortho", PAPER["wer_ortho"]),
                        ("CER", "cer", PAPER["cer"])):
    d = m_ck[key] - ref
    print(f"{label:<22}{m_ck[key]:>10.2f}{ref:>10.3f}{d:>+12.2f}{100*d/ref:>+10.1f}")

print(f"\n95% bootstrap CI on WER: {ci_ck[0]:.2f} - {ci_ck[1]:.2f}")
inside = ci_ck[0] <= PAPER["wer"] <= ci_ck[1]
print(f"the paper's 24.89% lies {'INSIDE' if inside else 'OUTSIDE'} our confidence interval"
      f"  ->  {'REPRODUCED' if inside else 'DISCREPANCY -- investigate and report it'}")
print(f"\nerrors: S {m_ck['sub']} / D {m_ck['del']} / I {m_ck['ins']}")
print(f"substitutions are {m_ck['sub_pct']}% of all word errors"
      f"   (the paper's error analysis reports ~70-78%)")
print(f"Pashto-script output rate: {m_ck['pashto_script_rate']:.2f}%"
      f"   (zero-shot measured above: {m_zs['pashto_script_rate']:.2f}%)")
print(f"WER/CER ratio: {m_ck['wer']/m_ck['cer']:.2f}   (paper: {PAPER['wer']/PAPER['cer']:.2f})")

p = paired_bootstrap(REFS, hyp_zs, hyp_ck)
print(f"\nfine-tuning improves WER by {m_zs['wer']-m_ck['wer']:.2f} pp, "
      f"paired bootstrap p = {p:.4f}")

pd.DataFrame({"reference": REFS, "hypothesis": hyp_ck}).to_csv(
    RESULTS / "predictions_author_checkpoint.csv", index=False, encoding="utf-8")

## 10. Part D — the orthographic-normalisation ablation (ours)

Re-scores the **identical predictions** under Pashto canonicalisation. No re-decoding, so this costs seconds.

Whatever reduction appears here is the share of the measured WER that was Arabic-script **spelling variation** rather than recognition error. These numbers are *not* comparable to the paper's 24.89 %, which uses the paper's normaliser — say so explicitly in the caption of Table 4.

In [ ]:
rows = []
for tag, hyp in (("zero-shot whisper-small", hyp_zs), ("author's exp_009 checkpoint", hyp_ck)):
    a = score(REFS, hyp)
    b = score(REFS, hyp, extra_norm=pashto_canonical)
    rows.append({"system": tag,
                 "wer_paper_normaliser": round(a["wer"], 2),
                 "wer_plus_pashto_canon": round(b["wer"], 2),
                 "wer_reduction_pp": round(a["wer"] - b["wer"], 2),
                 "cer_paper_normaliser": round(a["cer"], 2),
                 "cer_plus_pashto_canon": round(b["cer"], 2),
                 "cer_reduction_pp": round(a["cer"] - b["cer"], 2)})

abl = pd.DataFrame(rows)
abl.to_csv(RESULTS / "normalisation_ablation.csv", index=False, encoding="utf-8")
print(abl.to_string(index=False))
print(f"\nOn the author's own checkpoint, {rows[1]['wer_reduction_pp']} pp of the measured WER")
print("is attributable to legitimate Pashto spelling variation rather than to")
print("recognition error. This qualifies the paper's 24.89% without contradicting it:")
print("BasicTextNormalizer performs no Arabic-script canonicalisation, so some of the")
print("reported error is disagreement about orthography in the Common Voice references.")

## 11. Results table and the difference arithmetic

Writes `results/metrics.csv` with a `source` column holding exactly `paper` or `ours`, so the separation between the paper's numbers and ours is structural rather than editorial. Copy the printed block straight into **§7.1** of the report.

In [ ]:
import pandas as pd, pathlib, torch
RESULTS = pathlib.Path("results")

def row(tag, experiment, model, m, ci, paper_wer, paper_cer=None):
    d = {"tag": tag, "source": "ours", "experiment": experiment, "model": model,
         "split": f"cv24-test (N={m['n']})", "n_utterances": m["n"],
         "wer": round(m["wer"], 2), "wer_ortho": round(m["wer_ortho"], 2),
         "cer": round(m["cer"], 3),
         "wer_ci_lo": round(ci[0], 2), "wer_ci_hi": round(ci[1], 2),
         "paper_wer": paper_wer, "paper_cer": paper_cer,
         "difference_pp": round(m["wer"] - paper_wer, 2),
         "difference_rel_pct": round(100 * (m["wer"] - paper_wer) / paper_wer, 1),
         "substitutions": m["sub"], "deletions": m["del"], "insertions": m["ins"],
         "sub_pct_of_errors": m["sub_pct"], "del_pct_of_errors": m["del_pct"],
         "ins_pct_of_errors": m["ins_pct"],
         "pashto_script_rate": round(m["pashto_script_rate"], 2),
         "num_beams": 1, "precision": str(DTYPE).replace("torch.", ""),
         "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
         "eval_subsample": MAX_EVAL_SAMPLES or "full split", "seed": SEED}
    return d

metrics = pd.DataFrame([
    {"tag": "paper_zeroshot_small", "source": "paper", "experiment": "zero-shot",
     "model": "openai/whisper-small", "wer": PAPER["zeroshot_wer"]},
    {"tag": "paper_exp009", "source": "paper", "experiment": "fine-tuned (paper)",
     "model": "ihanif/exp_009_small_cv24_vanilla", "wer": PAPER["wer"],
     "wer_ortho": PAPER["wer_ortho"], "cer": PAPER["cer"]},
    row("zeroshot_small", "zero-shot", "openai/whisper-small",
        m_zs, ci_zs, PAPER["zeroshot_wer"]),
    row("author_checkpoint", "author checkpoint", CKPT,
        m_ck, ci_ck, PAPER["wer"], PAPER["cer"]),
])
metrics.to_csv(RESULTS / "metrics.csv", index=False, encoding="utf-8")

print("=" * 78)
print("SECTION 7.1 -- DIFFERENCE IN THE MAIN EVALUATION METRIC")
print("=" * 78)
print("Main metric: normalised WER on the Common Voice v24 Pashto test split.")
print("Sign convention: difference = ours - paper (positive means ours is worse).\n")
print("Part B  (author's released checkpoint, our harness)")
print(f"  WER_paper           = {PAPER['wer']:.2f} %")
print(f"  WER_ours            = {m_ck['wer']:.2f} %   95% CI [{ci_ck[0]:.2f}, {ci_ck[1]:.2f}]")
print(f"  Absolute difference = {m_ck['wer']:.2f} - {PAPER['wer']:.2f} = {m_ck['wer']-PAPER['wer']:+.2f} pp")
print(f"  Relative difference = {100*(m_ck['wer']-PAPER['wer'])/PAPER['wer']:+.1f} %")
print(f"  CER_paper = {PAPER['cer']:.3f} %   CER_ours = {m_ck['cer']:.2f} %   "
      f"difference = {m_ck['cer']-PAPER['cer']:+.2f} pp")
print("\nZero-shot baseline")
print(f"  WER_paper = {PAPER['zeroshot_wer']:.2f} %   WER_ours = {m_zs['wer']:.2f} %   "
      f"difference = {m_zs['wer']-PAPER['zeroshot_wer']:+.2f} pp")
print(f"\nImprovement from fine-tuning (ours, paired bootstrap)")
print(f"  {m_zs['wer']:.2f} - {m_ck['wer']:.2f} = {m_zs['wer']-m_ck['wer']:.2f} pp,  p = {p:.4f}")
print("\nScript-failure analysis (ours, not in the paper)")
print(f"  zero-shot Pashto-script output rate  = {m_zs['pashto_script_rate']:.2f} %")
print(f"  fine-tuned Pashto-script output rate = {m_ck['pashto_script_rate']:.2f} %")
print(f"  insertions as a share of zero-shot errors = {m_zs['ins_pct']:.1f} %")
print("=" * 78)
print(f"\nEvaluated on {m_ck['n']} utterances "
      f"({'FULL official split' if not MAX_EVAL_SAMPLES else f'seeded subsample of {n_full}'}).")
if MAX_EVAL_SAMPLES:
    print("Re-run with MAX_EVAL_SAMPLES = None before these numbers go in the report.")
metrics

## 12. Figures 3, 4, 5, 6 and the paper's own training curve

Paper bars are hatched and muted and labelled "Rahman (2026), reported"; ours are solid and labelled "This reproduction", so the two sources are distinguishable in the charts and not only in the prose. The WER axis starts at zero — a truncated axis would visually exaggerate a small gap.

The last figure is the paper's own exp_009 evaluation curve, reconstructed from the log recovered in cell 3. It is a figure from the original work that we reproduce, which is explicitly asked for in requirement (g), and it shows the loss/WER decoupling that justifies selecting checkpoints on WER.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, pandas as pd, pathlib

RESULTS = pathlib.Path("results")
PAPER_STYLE = dict(color="#9aa5b1", hatch="//", edgecolor="#4a5568", linewidth=1.2)
OURS_STYLE  = dict(color="#2b6cb0", edgecolor="#1a365d", linewidth=1.2)

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(RESULTS / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.close(fig); print("  ", name)

def label(ax, bars, fmt="{:.2f}"):
    for b in bars:
        ax.annotate(fmt.format(b.get_height()),
                    (b.get_x() + b.get_width()/2, b.get_height()),
                    ha="center", va="bottom", fontsize=9)

print("figures:")

# --- Figure 3: zero-shot vs fine-tuned -------------------------------
fig, ax = plt.subplots(figsize=(5.5, 4.5))
vals = [m_zs["wer"], m_ck["wer"]]
bars = ax.bar(["Zero-shot", "Fine-tuned"], vals, width=0.55, **OURS_STYLE)
bars[0].set_color("#c53030"); bars[0].set_edgecolor("#742a2a")
label(ax, bars, "{:.2f}%")
ax.axhline(100, ls=":", color="#718096", lw=1.2)
ax.text(1.46, 101.5, "100% = output nothing at all", fontsize=8, color="#718096", ha="right")
ax.annotate(f"-{vals[0]-vals[1]:.1f} pp", xy=(0.5, max(vals)*0.55), ha="center",
            fontsize=12, color="#1a365d", weight="bold")
ax.set_ylabel("WER (%)"); ax.set_ylim(0, max(vals)*1.2)
ax.set_title("Pashto WER: zero-shot vs fine-tuned whisper-small\nCommon Voice v24 test split")
ax.grid(axis="y", alpha=0.3)
save(fig, "wer_zeroshot_vs_finetuned")

# --- Figure 4: ours vs paper, WER ------------------------------------
# Two panels with independent y-axes. A single shared axis would be dominated
# by the ~120% zero-shot bars and squash the fine-tuned comparison -- which is
# the one the report actually turns on -- into illegibility. Both panels still
# start at zero, so neither exaggerates its gap.
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
panels = [("Zero-shot whisper-small", PAPER["zeroshot_wer"], m_zs["wer"], None),
          ("Fine-tuned whisper-small", PAPER["wer"], m_ck["wer"], ci_ck)]

for ax, (title, theirs, ours, ci) in zip(axes, panels):
    bp = ax.bar([0], [theirs], 0.5, label="Rahman (2026), reported", **PAPER_STYLE)
    bo = ax.bar([1], [ours],   0.5, label="This reproduction",       **OURS_STYLE)
    ax.annotate(f"{theirs:.2f}", (0, theirs), ha="center", va="bottom",
                fontsize=10, xytext=(0, 3), textcoords="offset points")
    if ci:
        ax.errorbar([1], [ours], yerr=[[ours-ci[0]], [ci[1]-ours]],
                    fmt="none", ecolor="#1a365d", capsize=5, lw=1.4)
        # clear the error bar, or the cap cuts through the value label
        ax.annotate(f"{ours:.2f}", (1, ci[1]), ha="center", va="bottom",
                    fontsize=10, xytext=(0, 4), textcoords="offset points")
    else:
        ax.annotate(f"{ours:.2f}", (1, ours), ha="center", va="bottom",
                    fontsize=10, xytext=(0, 3), textcoords="offset points")
    top = max(theirs, ours, ci[1] if ci else 0) * 1.38
    ax.set_ylim(0, top)
    # annotate at a fixed fraction of the axis, never relative to bar height,
    # or the delta label collides with the bar value labels on short bars
    ax.annotate(f"{ours-theirs:+.2f} pp\n({100*(ours-theirs)/theirs:+.1f} %)",
                (0.5, top * 0.88), ha="center", va="top",
                fontsize=10, color="#742a2a", weight="bold")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["paper", "ours"])
    ax.set_xlim(-0.6, 1.6)
    ax.set_title(title, fontsize=11)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("WER (%)")
axes[0].axhline(100, ls=":", color="#718096", lw=1.1)
axes[0].text(-0.55, 102, "100% = a system that outputs nothing", fontsize=7.5,
             color="#718096", ha="left", va="bottom")
axes[1].legend(frameon=False, fontsize=9, loc="upper right")
fig.suptitle("Our reproduction against the paper's reported WER\n"
             "whisper-small, Common Voice v24 Pashto test "
             f"(N={m_ck['n']})", y=1.04)
save(fig, "wer_comparison")

# --- Figure 5: CER ---------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 4.5))
bp = ax.bar([-0.18], [PAPER["cer"]], 0.36, label="Rahman (2026), reported", **PAPER_STYLE)
bo = ax.bar([ 0.18], [m_ck["cer"]],  0.36, label="This reproduction",       **OURS_STYLE)
label(ax, bp); label(ax, bo)
ax.annotate(f"{m_ck['cer']-PAPER['cer']:+.2f} pp",
            (0, max(PAPER["cer"], m_ck["cer"])*1.1), ha="center",
            fontsize=10, color="#742a2a", weight="bold")
ax.set_xticks([0]); ax.set_xticklabels(["Fine-tuned whisper-small"])
ax.set_ylabel("CER (%)"); ax.set_ylim(0, max(PAPER["cer"], m_ck["cer"])*1.35)
ax.set_title(f"CER comparison\nWER/CER ratio -- paper {PAPER['wer']/PAPER['cer']:.2f}, "
             f"ours {m_ck['wer']/m_ck['cer']:.2f}")
ax.legend(frameon=False); ax.grid(axis="y", alpha=0.3)
save(fig, "cer_comparison")

# --- Figure 6: error-type breakdown ----------------------------------
fig, ax = plt.subplots(figsize=(6, 4.5))
labels = ["Zero-shot", "Fine-tuned"]
series = {"Substitutions": [m_zs["sub_pct"], m_ck["sub_pct"]],
          "Deletions":     [m_zs["del_pct"], m_ck["del_pct"]],
          "Insertions":    [m_zs["ins_pct"], m_ck["ins_pct"]]}
colours = {"Substitutions": "#2b6cb0", "Deletions": "#d69e2e", "Insertions": "#c53030"}
bottom = [0, 0]
for name, vals in series.items():
    ax.bar(labels, vals, bottom=bottom, label=name, color=colours[name],
           edgecolor="white", width=0.5)
    bottom = [b + v for b, v in zip(bottom, vals)]
ax.axhline(74, ls="--", color="#4a5568", lw=1)
ax.text(1.45, 75.5, "paper: ~70-78% substitutions", fontsize=8, ha="right", color="#4a5568")
ax.set_ylabel("Share of total word errors (%)"); ax.set_ylim(0, 105)
ax.set_title("Error-type breakdown")
ax.legend(frameon=False, ncol=3, fontsize=8); ax.grid(axis="y", alpha=0.3)
save(fig, "error_breakdown")

# --- the paper's own exp_009 curve, reproduced ------------------------
curve = pd.read_csv(RESULTS / "paper_exp009_eval_curve.csv")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(curve["epoch"], curve["eval_wer"], marker="o", ms=4, lw=1.5,
        color="#c53030", label="eval WER (paper's exp_009)")
best_i = curve["eval_wer"].idxmin()
star = ax.scatter([curve.loc[best_i, "epoch"]], [curve.loc[best_i, "eval_wer"]],
                  marker="*", s=260, color="#742a2a", zorder=5,
                  label=f"paper's best: {curve.loc[best_i,'eval_wer']:.2f}% "
                        f"at epoch {curve.loc[best_i,'epoch']:.2f}")
ax.axhline(m_ck["wer"], ls="--", color="#2b6cb0", lw=1.2,
           label=f"our measurement of the released checkpoint ({m_ck['wer']:.2f}%)")
ax.set_xlabel("Epoch"); ax.set_ylabel("WER (%)", color="#c53030")
ax.tick_params(axis="y", colors="#c53030")
tw = ax.twinx()
tw.plot(curve["epoch"], curve["eval_loss"], marker="s", ms=3, lw=1.2,
        color="#4a5568", alpha=0.75, label="eval loss")
tw.set_ylabel("Validation loss", color="#4a5568"); tw.tick_params(axis="y", colors="#4a5568")
lo_i = curve["eval_loss"].idxmin()
tw.axvline(curve.loc[lo_i, "epoch"], ls=":", color="#4a5568", lw=1,
           label=f"lowest loss at epoch {curve.loc[lo_i,'epoch']:.2f}")
# scatter is not a Line2D, so add it to the legend explicitly or it vanishes
handles = ax.get_lines() + [star] + tw.get_lines()
ax.legend(handles, [h.get_label() for h in handles],
          frameon=True, framealpha=0.9, fontsize=7.5, loc="upper right")
ax.set_title("The paper's own exp_009 training curve, reconstructed from its published log\n"
             "loss rises after epoch ~2.8 while WER keeps improving")
ax.grid(alpha=0.3)
save(fig, "paper_exp009_curve_reproduced")

print("\nall figures written to results/ as 300 dpi PNG and PDF")

## 13. Sample predictions for §8.1

Twenty utterances drawn from the best, median and worst terciles of per-utterance WER, so the selection is demonstrably not cherry-picked.

**When you paste this into Word, set the three transcript columns to Noto Naskh Arabic and mark them right-to-left**, or the Pashto will print as empty boxes.

In [ ]:
import jiwer, pandas as pd, pathlib
RESULTS = pathlib.Path("results")

per = []
for ref, a, b in zip(REFS, hyp_zs, hyp_ck):
    rn = _norm(ref).strip()
    if not rn:
        continue
    o = jiwer.process_words(rn, _norm(b).strip() or " ")
    per.append({"reference": ref, "zero_shot_hypothesis": a[:200],
                "finetuned_hypothesis": b,
                "wer_utt": round(100 * o.wer, 1),
                "n_sub": o.substitutions, "n_del": o.deletions, "n_ins": o.insertions})

per = pd.DataFrame(per).sort_values("wer_utt").reset_index(drop=True)
k, mid = 7, len(per) // 2
sample = pd.concat([
    per.head(k).assign(bucket="best"),
    per.iloc[mid - 3: mid + 4].assign(bucket="median"),
    per.tail(k).assign(bucket="worst"),
])
sample.to_csv(RESULTS / "sample_predictions.csv", index=False, encoding="utf-8")
per.to_csv(RESULTS / "per_utterance_metrics.csv", index=False, encoding="utf-8")
print(f"wrote results/sample_predictions.csv ({len(sample)} rows) "
      f"and results/per_utterance_metrics.csv ({len(per)} rows)")
print(f"\nper-utterance WER distribution (fine-tuned): "
      f"min {per.wer_utt.min():.1f}  median {per.wer_utt.median():.1f}  "
      f"mean {per.wer_utt.mean():.1f}  max {per.wer_utt.max():.1f}")
print(f"utterances transcribed perfectly (WER 0): "
      f"{(per.wer_utt == 0).sum()} of {len(per)} ({100*(per.wer_utt==0).mean():.1f}%)")
sample[["bucket", "wer_utt", "reference", "finetuned_hypothesis"]].head(21)

## 14. Save everything and download

Zips `results/` so you can pull one file out of Colab and commit it to the repository. Everything in it is required evidence for the report.

### What to screenshot before you close this notebook
The report has eight screenshot boxes. Five of them come from this notebook:

| Screenshot | Cell |
|---|---|
| 1 — environment and `nvidia-smi` | cells 1–2 |
| 2 — dataset preparation, row counts against the paper's 13,791, filter table | cell 5 |
| 3 — zero-shot evaluation, WER above 100 %, script-failure rate | cell 8 |
| 4 — the author's checkpoint evaluated, difference from 24.89 % printed | cell 9 |
| 7 — the §7.1 difference block and the figures | cells 11–12 |

Keep the timestamps visible. Do not crop or edit the outputs.

In [ ]:
import shutil, pathlib, os, sys
RESULTS = pathlib.Path("results")

print("files produced:")
for p in sorted(RESULTS.rglob("*")):
    if p.is_file():
        print(f"  {p.stat().st_size:>12,}  {p.relative_to(RESULTS)}")

shutil.make_archive("batch3_results", "zip", RESULTS)
print(f"\nbatch3_results.zip  {os.path.getsize('batch3_results.zip'):,} bytes")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download("batch3_results.zip")
else:
    print("Kaggle: the file is in /kaggle/working -- use the Output tab to download it.")

## 15. What to do with these numbers

Open `report/CS444_A2_PashtoWhisper_Report.docx` and fill these placeholders. Everything below is now a measured number, not a guess.

| Report location | Fill from |
|---|---|
| Abstract, §6.2 Table 1, §7 Table 5, §7.1, §9.1(a), §10 | cell 9 — the author-checkpoint WER and CER, and the difference from 24.89 % |
| §6.2 Table 1, §7, §7.1, §9.3, Abstract | cell 8 — zero-shot WER |
| §6.4 Table 3 | cells 8–9 — Pashto-script output rates and insertion shares |
| §6.4 Table 4 | cell 10 — the normalisation ablation |
| §4.2 test rows, hours, speakers | cell 5 |
| Appendix B | `results/preprocessing_log_batch3.csv` |
| §5.1 versions, GPU | `results/environment.txt`, `results/gpu.txt` |
| §8 Figures 3, 4, 5, 6 and the reproduced paper figure | `results/*.png` |
| §8.1 sample predictions | `results/sample_predictions.csv` |
| Appendix A.1 log excerpts | the printed output of cells 5, 8, 9, 11 |

### Three edits to the report that cell 3 makes necessary

Cell 3 recovered configuration that the paper does not publish, which changes several rows you were going to mark "assumed".

1. **§5.3 deviation table.** Replace the "not stated / assumed" entries with the recovered values: learning rate 1e-4, `train_batchsize` 64 with `gradient_accumulation_steps` 1, FP32 (`fp16: False`, `bf16: False`), `warmup_steps` 500, `weight_decay` 0.0, `early_stopping_patience` 5, `num_epochs` 15, `eval_steps` 500. Keep the optimizer row as an inference, but strengthen the wording: no `optim` argument appears in the published configuration, so the `Seq2SeqTrainer` default `adamw_torch` was used. Cite `experiment_log.txt` in the author's checkpoint repository as the source.

2. **§9 Discussion — add a paragraph.** `eval_dataset_splits: ['test']`. The evaluation loop that selected the best checkpoint ran on the **test split**, so the paper's reported 24.89 % is selected on the same data it is reported on. The paper's prose calls this "validation WER". This is an optimistic bias of unknown size, and it makes our held-out-dev design methodologically cleaner than the paper's. State it factually, without editorialising — it is a finding, not an accusation.

3. **§4.4 preprocessing.** The author's duration floor is 0.5 s, not the 1.0 s in our `prepare_data.py`. Either change the config to 0.5 to match, or keep 1.0 and record it in the deviation table. Matching is easier to defend.

### Then, optionally
Re-run cell 4 with `MAX_EVAL_SAMPLES = None` for the final full-split numbers, and go on to Batch 4 (`src/train.py`) for your own training run. Batch 3 alone already gives you a complete, honest report with §6.3 marked as not completed in the Limitations.